In [5]:
import pandas as pd

df = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(541910, 8)
  Invoice StockCode                          Description  Quantity  \
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                  WHITE METAL LANTERN         6   
2  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  Price  Customer ID         Country  
0 2010-12-01 08:26:00   2.55      17850.0  United Kingdom  
1 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
2 2010-12-01 08:26:00   2.75      17850.0  United Kingdom  
3 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
4 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float

In [6]:
print(f"Before dropping nulls: {df.shape}")
df = df.dropna(subset=['Customer ID'])
print(f"After dropping nulls: {df.shape}")
print(f"Rows removed: {541910 - df.shape[0]}")

Before dropping nulls: (541910, 8)
After dropping nulls: (406830, 8)
Rows removed: 135080


In [7]:
df['Customer ID'] = df['Customer ID'].astype(int)
print(df['Customer ID'].dtype)
print(df['Customer ID'].head())

int64
0    17850
1    17850
2    17850
3    17850
4    17850
Name: Customer ID, dtype: int64


In [8]:
print(f"Unique invoice prefixes sample: {df['Invoice'].astype(str).str[0].unique()}")
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Cancelled orders: {cancelled.shape[0]}")
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f"After removing cancellations: {df.shape}")

Unique invoice prefixes sample: ['5' 'C']
Cancelled orders: 8905
After removing cancellations: (397925, 8)


In [9]:
print(f"Rows with Price <= 0: {df[df['Price'] <= 0].shape[0]}")
print(f"Rows with Quantity <= 0: {df[df['Quantity'] <= 0].shape[0]}")
df = df[df['Price'] > 0]
df = df[df['Quantity'] > 0]
print(f"After removing bad rows: {df.shape}")

Rows with Price <= 0: 40
Rows with Quantity <= 0: 0
After removing bad rows: (397885, 8)


In [10]:
df['Revenue'] = df['Price'] * df['Quantity']
print(df[['Price', 'Quantity', 'Revenue']].head(10))
print(f"\nTotal revenue in dataset: £{df['Revenue'].sum():,.2f}")

   Price  Quantity  Revenue
0   2.55         6    15.30
1   3.39         6    20.34
2   2.75         8    22.00
3   3.39         6    20.34
4   3.39         6    20.34
5   7.65         2    15.30
6   4.25         6    25.50
7   1.85         6    11.10
8   1.85         6    11.10
9   4.25         6    25.50

Total revenue in dataset: £8,911,425.90


In [11]:
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Total unique customers: {df['Customer ID'].nunique()}")
print(f"Total unique invoices: {df['Invoice'].nunique()}")
print(f"Final clean dataset shape: {df.shape}")

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Total unique customers: 4338
Total unique invoices: 18532
Final clean dataset shape: (397885, 9)


In [12]:
print(df['Customer ID'].dtype)

int64


In [13]:
import datetime
snapshot_date = df['InvoiceDate'].max() + datetime.timedelta(days=1)
print(f"Snapshot date: {snapshot_date}")

Snapshot date: 2011-12-10 12:50:00


In [14]:
rfm = df.groupby('Customer ID').agg(
    recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    frequency=('Invoice', 'nunique'),
    monetary=('Revenue', 'mean')
).reset_index()

print(rfm.shape)
print(rfm.head(10))
print(rfm.describe())

(4338, 4)
   Customer ID  recency  frequency      monetary
0        12346      326          1  77183.600000
1        12347        2          7     23.681319
2        12348       75          4     57.975484
3        12349       19          1     24.076027
4        12350      310          1     19.670588
5        12352       36          8     29.482824
6        12353      204          1     22.250000
7        12354      232          1     18.610345
8        12355      214          1     35.338462
9        12356       23          3     47.651356
        Customer ID      recency    frequency      monetary
count   4338.000000  4338.000000  4338.000000   4338.000000
mean   15300.408022    92.536422     4.272015     68.350512
std     1721.808492   100.014169     7.697998   1467.918896
min    12346.000000     1.000000     1.000000      2.101286
25%    13813.250000    18.000000     1.000000     12.365367
50%    15299.500000    51.000000     2.000000     17.723119
75%    16778.750000   142.00000

In [15]:
print(f"Customers with frequency = 1: {rfm[rfm['frequency'] == 1].shape[0]}")
print(f"Customers with frequency > 10: {rfm[rfm['frequency'] > 10].shape[0]}")
print(f"Average recency (days): {rfm['recency'].mean():.0f}")
print(f"Average monetary (£): {rfm['monetary'].mean():.2f}")
print(f"Max frequency: {rfm['frequency'].max()}")

Customers with frequency = 1: 1493
Customers with frequency > 10: 337
Average recency (days): 93
Average monetary (£): 68.35
Max frequency: 209


In [16]:
df.to_csv('../data/processed/clean_transactions.csv', index=False)
rfm.to_csv('../data/processed/rfm_table.csv', index=False)
print("Saved successfully")

Saved successfully
